# 🏥 ViBioMIR Interactive Search & Document Inspection (Local & Kaggle)

Notebook này được tối ưu để chạy mượt mà trên **máy Local** (với môi trường `.venv`) lẫn trên **Kaggle / Colab**:
1. **Tự động xuống dòng (Text Wrap)**: Toàn bộ output dài đều được wrap gọn gàng, không bị tràn thanh cuộn ngang.
2. **Tra cứu toàn văn từ `doc_id`**: Tra cứu bất kỳ `doc_id` nào để lấy tiêu đề, toàn văn tài liệu và danh sách tất cả các chunks tạo nên tài liệu đó.
3. **Thử nghiệm tìm kiếm tương tác (Interactive Query Search)**: Tra cứu nhanh câu hỏi lâm sàng bằng Hybrid (FAISS SQ8 + BM25s + BGE-Reranker).
4. **Kiểm tra Top 1 F2-Macro Strategy**: Trích xuất gọn 4 docs x 1 chunk để tối ưu điểm $F_2$.
5. **Chạy trích xuất toàn bộ 1,200 câu hỏi** và xuất file `submission_top1_strategy.zip`.

In [9]:
# 1. Kích hoạt Tự động Xuống dòng (Text Wrapping) & Kiểm tra Thư viện
import sys
from IPython.display import HTML, display

# Kích hoạt CSS wrap text cho toàn bộ output trong Jupyter / VS Code / Kaggle
display(HTML("""
<style>
    div.output_area pre, div.output_subarea pre, .output_text pre, .output_area, .output_subarea {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
        word-break: break-word !important;
        overflow-x: hidden !important;
    }
</style>
"""))

try:
    import faiss
    import pyvi
    import bm25s
    import flagembedding
    from sentence_transformers import SentenceTransformer
    print("✅ Môi trường hiện tại đã cài đặt đầy đủ tất cả thư viện cần thiết!")
except ImportError as e:
    print(f"⚠️ Thiếu thư viện ({e}). Đang tiến hành cài đặt...")
    !{sys.executable} -m pip install -q faiss-gpu pyarrow "huggingface_hub[cli]" bm25s rank-bm25 pyvi jieba loguru pydantic pyyaml tqdm flagembedding sentence-transformers

⚠️ Thiếu thư viện (No module named 'flagembedding'). Đang tiến hành cài đặt...
/home/thienhb/Workspace/med-doc-retrieval/.venv/bin/python: No module named pip


In [10]:
# 2. Cấu hình Thư mục Gốc Dự án (Project Root) và Python Path
import os
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
if (current_dir / "src").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").exists():
    PROJECT_ROOT = current_dir.parent
elif Path("/kaggle/working/med-doc-retrieval").exists():
    PROJECT_ROOT = Path("/kaggle/working/med-doc-retrieval")
else:
    PROJECT_ROOT = current_dir

# Chuyển Working Directory về thư mục gốc của dự án
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("📁 Current Working Directory:", os.getcwd())
print("🐍 Python Executable:", sys.executable)

📁 Current Working Directory: /home/thienhb/Workspace/med-doc-retrieval
🐍 Python Executable: /home/thienhb/Workspace/med-doc-retrieval/.venv/bin/python


In [11]:
# 3. Kiểm tra Index sẵn có trên máy (FAISS SQ8 + BM25s + SQLite Slim)
from pathlib import Path
from huggingface_hub import HfApi

indices_dir = Path("data/indices")
indices_dir.mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

# Danh sách file cần thiết cho Hybrid Retrieval (FAISS SQ8 + SQLite + BM25s)
sync_files = [
    ("indices/dense_index_sq8.faiss", "data/indices/dense_index_sq8.faiss"),
    ("indices/chunks_meta_slim.sqlite", "data/indices/chunks_meta_slim.sqlite"),
    ("indices/dense_metadata_sq8.json", "data/indices/dense_metadata_sq8.json"),
    ("indices/bm25s_index/data.csc.index.npy", "data/indices/bm25s_index/data.csc.index.npy"),
    ("indices/bm25s_metadata.json", "data/indices/bm25s_metadata.json"),
    ("processed/queries_enriched.jsonl", "data/processed/queries_enriched.jsonl"),
]

missing = [(rem, loc) for rem, loc in sync_files if not Path(loc).exists() or Path(loc).stat().st_size == 0]

if missing:
    print(f"📥 Phát hiện {len(missing)} file chưa có, bắt đầu đồng bộ từ Hugging Face Bucket 'nieths/ViBioMIR'...")
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    api = HfApi(token=HF_TOKEN or None)
    api.download_bucket_files(bucket_id="nieths/ViBioMIR", files=missing, token=HF_TOKEN or None)
    print("✅ Hoàn tất tải file index!")
else:
    print("✅ Toàn bộ file index (SQ8 FAISS + BM25s + SQLite Slim) đã có sẵn trên máy!")

# Liệt kê các file trong thư mục index
for p in sorted(indices_dir.iterdir()):
    if p.is_file():
        print(f"  • {p.name:<30} ({p.stat().st_size / (1024*1024):.2f} MB)")
    elif p.is_dir():
        print(f"  📁 {p.name}/")

✅ Toàn bộ file index (SQ8 FAISS + BM25s + SQLite Slim) đã có sẵn trên máy!
  • .gitkeep                       (0.00 MB)
  📁 bm25s_index/
  • bm25s_metadata.json            (21.72 MB)
  • chunks_meta_slim.sqlite        (5173.18 MB)
  • dense_index_sq8.faiss          (2264.21 MB)
  • dense_metadata_sq8.json        (0.00 MB)
  📁 qdrant_db/


In [12]:
# 4. Khởi tạo Pipeline & Load Index (Hybrid: FAISS SQ8 + BM25s + SQLite)
from src.config import load_config
from src.pipeline import MedicalRetrievalPipeline

config = load_config("configs/config.yaml")
config.pubmed.source_api = "none"  # Tắt HTTP API ngoài, dùng cache local để inference nhanh nhất

pipeline = MedicalRetrievalPipeline(config=config)
pipeline.load_indices(
    indices_dir="data/indices",
    dense_dir="data/indices",
    sparse_dir="data/indices",
    chunks_meta_path="data/indices/chunks_meta_slim.sqlite",
)
print("🚀 Pipeline Hybrid (FAISS SQ8 + BM25s) đã khởi tạo thành công!")

2026-10-24 15:41:44.139 | INFO     | src.embedding.bge_m3:__init__:33 - Initializing BGEM3Embedder with model='BAAI/bge-m3' on device='cuda', fp16=True, return_sparse=True
2026-10-24 15:41:44.140 | INFO     | src.reranker.bge_reranker:__init__:28 - Initializing BGEReranker with model='BAAI/bge-reranker-v2-m3' on device='cuda', fp16=True
2026-10-24 15:41:44.164 | DEBUG    | src.crawler.query_translator:__init__:156 - Loaded 9440 terms from data/lexicon/medical_terms.json. Total: 9440
2026-10-24 15:41:44.177 | DEBUG    | src.crawler.query_translator:__init__:163 - Loaded 9076 Chinese medical terms from data/lexicon/icd10_vi_zh.json.
2026-10-24 15:41:44.179 | DEBUG    | src.crawler.query_translator:__init__:170 - Loaded 42 medical acronyms from data/lexicon/medical_acronyms.json.
2026-10-24 15:41:44.180 | DEBUG    | src.crawler.query_translator:__init__:177 - Loaded 80 stopwords from configs/pubmed_stopwords.txt. Total: 80
2026-10-24 15:41:44.483 | INFO     | src.crawler.pubmed:_load_cach

🚀 Pipeline Hybrid (FAISS SQ8 + BM25s) đã khởi tạo thành công!


### 📖 5. Tra cứu Toàn văn Tài liệu & Tất cả Chunks từ `doc_id`
Nhập bất kỳ `doc_id` nào để xem toàn văn tài liệu và danh sách tất cả các chunks tạo nên tài liệu đó.

In [21]:
from src.retrieval.doc_reader import display_document, get_document_and_chunks, print_document

# Thử tra cứu một doc_id mẫu (ví dụ: 545347 hoặc bất kỳ doc_id nào từ kết quả tìm kiếm)
target_doc_id = 5130738

# 1. Hiển thị dạng giao diện đẹp mắt (có wrap text tự động, hiển thị từng chunk rõ ràng):
display_document(target_doc_id)

# 2. Hoặc nếu muốn lấy dữ liệu dạng Dictionary thuần Python:
# doc_data = get_document_and_chunks(target_doc_id)
# print("Tiêu đề:", doc_data['title'])
# print("Số lượng chunks:", doc_data['total_chunks'])
# print("Toàn văn tài liệu:\n", doc_data['full_text'][:500])

### 🎲 6. UI Trực Quan: Đọc Dữ Liệu Cào Về Ngẫu Nhiên & So Sánh Chunks
Chạy cell dưới đây để **lấy ngẫu nhiên một tài liệu y khoa đã cào về** (`data/raw`) và quan sát song song:
- **Cột trái**: Toàn văn văn bản gốc cào về (kèm URL nguồn, số từ, độ dài ký tự)
- **Cột phải**: Danh sách các Chunks thực tế được cắt ra (đã wrap text gọn gàng)

> **Mẹo**: Nếu muốn dùng giao diện Web riêng biệt (có thanh trượt thử nghiệm `chunk_size` và `chunk_overlap`), hãy mở Terminal và chạy: `streamlit run apps/doc_chunk_viewer.py`

In [ ]:
from src.retrieval.doc_reader import display_random_crawled_doc

# Mỗi lần chạy lại cell này sẽ hiển thị một tài liệu cào về ngẫu nhiên mới:
# Bạn có thể truyền target_lang='vi' hoặc target_lang='zh' nếu muốn lọc ngôn ngữ cụ thể:
display_random_crawled_doc(target_lang=None)

### 🔍 7. Tìm kiếm Tương tác (Interactive Search) & Trực quan hóa Bằng chứng
Truy vấn một câu hỏi lâm sàng bất kỳ, xem Top documents và xem toàn văn tài liệu đứng đầu:

In [20]:
test_query = "Bệnh nhân bị xuất huyết giảm tiểu cầu vô căn điều trị bằng thuốc gì đầu tay?"
test_query = "Thuốc kích thích HGH có làm tăng tế bào ung thư không?"

print(f"🔍 Đang tìm kiếm cho câu hỏi: '{test_query}'\n")
result = pipeline.search_query(test_query)

print("--- 🏆 TOP TÀI LIỆU LIÊN QUAN (DOC_IDS) ---")
for rank, did in enumerate(result["relevant_docs"][:5], 1):
    print(f"  #{rank} Doc ID: {did}")

print("\n--- 📑 TOP CHUNKS BẰNG CHỨNG TRÍCH XUẤT (ĐÃ WRAP TEXT) ---")
for rank, ch in enumerate(result["relevant_chunks"][:3], 1):
    print(f"  #{rank} [Doc ID {ch['doc_id']}]:")
    print(f"     \"{ch['chunk_text'].strip()}\"\n")

# Hiển thị toàn văn tài liệu #1 vừa tìm được:
if result["relevant_docs"]:
    top_1_id = result["relevant_docs"][0]
    print(f"\n👉 Chi tiết Toàn văn Tài liệu xếp hạng #1 (Doc ID: {top_1_id}):")
    display_document(top_1_id)

🔍 Đang tìm kiếm cho câu hỏi: 'Thuốc kích thích HGH có làm tăng tế bào ung thư không?'



--- 🏆 TOP TÀI LIỆU LIÊN QUAN (DOC_IDS) ---
  #1 Doc ID: 5130738
  #2 Doc ID: 650657
  #3 Doc ID: 109512
  #4 Doc ID: 93646
  #5 Doc ID: 5698056

--- 📑 TOP CHUNKS BẰNG CHỨNG TRÍCH XUẤT (ĐÃ WRAP TEXT) ---
  #1 [Doc ID 5130738]:
     "In a recent study we demonstrated that recombinant human growth hormone (r-hGH; Saizen) delayed tumor-induced cachexia in human tumor xenografts in vivo. Such a therapeutic effect could have a great impact in the supportive care of advanced cancer patients. Before large clinical studies are initiated possible growth stimulation should be excluded. This question was investigated in vitro in 20 human tumor models, which had been established in serial passage in nude mice. The effect of continuous exposure of r-hGH was investigated at dose levels ranging from 0.3 ng/ml up to 0.1 microg/ml in colorectal (n=2), gastric (n=1), non-small cell lung (n=4), small cell lung (n=1), mammary (n=3), ovarian (n=2), prostate (n=2) and renal cancers (n=2), and melanoma (n=3) 

In [ ]:
# 7. Chạy thử nghiệm trên 5 query ngẫu nhiên trong tập test
import json
import pyarrow.parquet as pq

queries_path = Path("data/processed/queries_enriched.jsonl")
sample_queries = []

if queries_path.exists():
    with open(queries_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5: break
            sample_queries.append(json.loads(line))
elif Path("data/raw/vibio_mir/query.parquet").exists():
    table = pq.read_table("data/raw/vibio_mir/query.parquet")
    sample_queries = table.to_pylist()[:5]

print(f"Chạy kiểm tra trên {len(sample_queries)} câu hỏi ngẫu nhiên:\n")
for q in sample_queries:
    qid = q["id"]
    q_text = q.get("query") or q.get("original_query")
    res = pipeline.search_query(q_text)
    print(f"[Query #{qid}]: {q_text}")
    print(f"  -> Top Docs: {res['relevant_docs'][:4]}")
    print(f"  -> Số Chunks trích xuất: {len(res['relevant_chunks'])}\n")

In [ ]:
# 8. Chạy Inference toàn bộ tập test với CHIẾN LƯỢC TOP 1 (Tối ưu Macro F2)
# Áp dụng: top_k_docs=4, top_k_chunks=4, max_chunks_per_doc=1 để tránh phạt rác Chunk Precision
import sys

!{sys.executable} scripts/run_inference_only.py \
    --indices-dir "data/indices" \
    --dense-dir "data/indices" \
    --sparse-dir "data/indices" \
    --chunks-meta-path "data/indices/chunks_meta_slim.sqlite" \
    --hybrid-top-k 200 \
    --top-k-docs 4 \
    --top-k-chunks 4 \
    --max-chunks-per-doc 1 \
    --batch-size 16 \
    --disable-pubmed-network \
    --output-name submission_top1_strategy.json

In [ ]:
# 9. Kiểm tra File Submission đã sinh & Tải về máy
import base64
from IPython.display import HTML, display

sub_dirs = [Path("outputs/submissions"), Path("/kaggle/working"), Path(".")]
zip_candidates = []
for d in sub_dirs:
    if d.exists():
        zip_candidates.extend(list(d.glob("*.zip")))

if zip_candidates:
    latest_zip = sorted(zip_candidates, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    size_mb = latest_zip.stat().st_size / (1024 * 1024)
    print(f"🎉 File submission mới nhất: {latest_zip.resolve()} ({size_mb:.2f} MB)")
    
    with open(latest_zip, "rb") as f:
        data_b64 = base64.b64encode(f.read()).decode()
    
    html_button = f'''
    <div style="padding:15px; background:#e8f4fd; border:2px solid #2196f3; border-radius:8px; margin-top:10px;">
       <h4 style="color:#0b5394; margin:0 0 8px 0;">📦 Tải Submission ZIP ({latest_zip.name})</h4>
       <a href="data:application/zip;base64,{data_b64}" download="{latest_zip.name}" 
          style="display:inline-block; padding:10px 20px; background:#2196f3; color:white; 
                 text-decoration:none; border-radius:6px; font-weight:bold; font-size:14px;">
          ⬇ BẤM VÀO ĐÂY ĐỂ TẢI VỀ MÁY
       </a>
    </div>
    '''
    display(HTML(html_button))
else:
    print("Chưa tìm thấy file submission .zip nào. Hãy chạy cell 8 trước!")